# Ukraine TikTok dataset

This notebook analyses the **publicly downloadable release** of *The Invasion of Ukraine Viewed through TikTok*. It keeps ID analysis separate from comments. The official public package contains IDs and reconstruction code, not the 4.4 million comment rows described in the paper.

Sources: [official repository](https://github.com/networkdynamics/ukraine-tiktok), [paper](https://arxiv.org/abs/2301.08305), and [PyTok](https://github.com/networkdynamics/pytok).

## 1. Load and verify the local official files

In [1]:
from pathlib import Path
import hashlib
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'downloaded_datasets' / 'ukraine'

FILES = {
    'filtered_video_ids': DATA_DIR / 'video_ids.csv',
    'raw_video_ids': DATA_DIR / 'raw_video_ids.csv',
    'manual_coding': DATA_DIR / 'video_coding.csv',
    'official_archive': DATA_DIR / 'ukraine_tiktok_dataset.zip',
}
for path in FILES.values():
    if not path.exists():
        raise FileNotFoundError(path)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

inventory = pd.DataFrame([
    {'file': path.name, 'bytes': path.stat().st_size, 'MiB': path.stat().st_size / 1024**2, 'sha256': sha256(path)}
    for path in FILES.values()
])
display(inventory.round({'MiB': 3}))

with zipfile.ZipFile(FILES['official_archive']) as archive:
    archive_contents = pd.DataFrame([
        {'file_in_archive': item.filename, 'uncompressed_bytes': item.file_size}
        for item in archive.infolist() if not item.is_dir()
    ])
display(archive_contents)

,file,bytes,MiB,sha256
0,video_ids.csv,315343,0.30,d50a63b62f2641cdfda45c7af313c7030d245ae57deb77...
1,raw_video_ids.csv,621886,0.59,7bc86632418cadf4cb92a85017127033380bbf576b3352...
2,video_coding.csv,24456,0.02,1228ae1ef3d310c19aeb0fc3cf088070529cd6e526d608...
3,ukraine_tiktok_dataset.zip,474809,0.45,e96977c4437077bf985bb3377d677e77ec3ab07f264985...


,file_in_archive,uncompressed_bytes
0,additional_data/raw_video_ids.csv,621886
1,additional_data/video_coding.csv,24456
2,get_videos.py,2075
3,hashtag_tiktoks.py,1318
4,load_json_to_csv.py,1339
5,README.md,7128
6,search_tiktoks.py,1485
7,user_tiktoks.py,2400
8,video_comments.py,2189
9,video_ids.csv,315343


## 2. Structure of the released data

In [2]:
filtered_ids = pd.read_csv(FILES['filtered_video_ids'], dtype='string')
raw_ids = pd.read_csv(FILES['raw_video_ids'], dtype='string')
coding = pd.read_csv(FILES['manual_coding'], dtype='string')

schema = pd.DataFrame([
    {'file': 'video_ids.csv', 'rows': len(filtered_ids), 'columns': ', '.join(filtered_ids.columns)},
    {'file': 'raw_video_ids.csv', 'rows': len(raw_ids), 'columns': ', '.join(raw_ids.columns)},
    {'file': 'video_coding.csv', 'rows': len(coding), 'columns': ', '.join(coding.columns)},
])
display(schema)
display(filtered_ids.head(3))
display(coding.head(3))

released_content = pd.DataFrame([
    {'content': 'TikTok video ID', 'public_release': 'Yes'},
    {'content': 'Creator username', 'public_release': 'Yes'},
    {'content': 'Manual relevance coding sample', 'public_release': 'Yes'},
    {'content': 'Video metadata and engagement counters', 'public_release': 'No; hydration required'},
    {'content': 'Comment rows or comment text', 'public_release': 'No; hydration required'},
    {'content': 'Raw video, audio, transcript, OCR', 'public_release': 'No'},
])
display(released_content)

,file,rows,columns
0,video_ids.csv,9503,"video_id, user_unique_id"
1,raw_video_ids.csv,18864,"video_id, user_unique_id"
2,video_coding.csv,392,"video_id, author_name, my_desc, related"


,video_id,user_unique_id
0,7068563399923322117,dylanpage.ning
1,7070499639832661254,laminaprod
2,7068310189472042266,hannanajj


,video_id,author_name,my_desc,related
0,7090247339964665093,mynezalezhna,Zelensky support,Y
1,7076397333730282798,ret.navy.wife,Support for NATO to help ukraine in war,Y
2,7075253619737513222,wellestt,Support for Zelensky,Y


,content,public_release
0,TikTok video ID,Yes
1,Creator username,Yes
2,Manual relevance coding sample,Yes
3,Video metadata and engagement counters,No; hydration required
4,Comment rows or comment text,No; hydration required
5,"Raw video, audio, transcript, OCR",No


## 3. Check the release counts and ID quality

In [3]:
checks = pd.DataFrame([
    {
        'release': 'Filtered IDs',
        'rows': len(filtered_ids),
        'unique_video_ids': filtered_ids['video_id'].nunique(),
        'duplicate_id_rows': int(filtered_ids['video_id'].duplicated().sum()),
        'unique_creator_usernames': filtered_ids['user_unique_id'].nunique(),
    },
    {
        'release': 'Raw pre-filter IDs',
        'rows': len(raw_ids),
        'unique_video_ids': raw_ids['video_id'].nunique(),
        'duplicate_id_rows': int(raw_ids['video_id'].duplicated().sum()),
        'unique_creator_usernames': raw_ids['user_unique_id'].nunique(),
    },
    {
        'release': 'Manual coding rows',
        'rows': len(coding),
        'unique_video_ids': coding['video_id'].nunique(),
        'duplicate_id_rows': int(coding['video_id'].duplicated().sum()),
        'unique_creator_usernames': coding['author_name'].nunique(),
    },
])
display(checks)

print('All filtered IDs also occur in the raw release:', bool(filtered_ids['video_id'].isin(set(raw_ids['video_id'])).all()))
print('Filtered share of raw IDs:', f"{len(filtered_ids) / len(raw_ids) * 100:.2f}%")

,release,rows,unique_video_ids,duplicate_id_rows,unique_creator_usernames
0,Filtered IDs,9503,9503,0,5782
1,Raw pre-filter IDs,18864,18864,0,12120
2,Manual coding rows,392,388,4,279


All filtered IDs also occur in the raw release: True
Filtered share of raw IDs: 50.38%


## 4. Interesting point: creator concentration

In [4]:
videos_per_creator = filtered_ids.groupby('user_unique_id').size().rename('videos')
creator_summary = videos_per_creator.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]).to_frame().T
display(creator_summary)
display(videos_per_creator.sort_values(ascending=False).head(15).to_frame())

top_1_percent_n = max(1, int(np.ceil(len(videos_per_creator) * 0.01)))
top_1_percent_share = videos_per_creator.nlargest(top_1_percent_n).sum() / videos_per_creator.sum() * 100
print('Share of released videos from the top 1% of creators:', f'{top_1_percent_share:.2f}%')

,count,mean,std,min,50%,75%,90%,95%,99%,max
videos,"5,782.00",1.64,3.42,1.00,1.00,1.00,2.00,4.00,13.00,151.00


,videos
user_unique_id,
news_from_di_kz,151
skynews,72
zelenskyfan32,70
valerisssh,52
philipdefranco,50
vsp.zelensky,38
mynezalezhna,36
cbsmornings,35
viceworldnews,35


Share of released videos from the top 1% of creators: 15.66%


## 5. Interesting point: the relevance filter

In [5]:
coding['related_clean'] = coding['related'].str.strip()
coding_counts = coding['related_clean'].value_counts(dropna=False).rename('rows').to_frame()
coding_counts['percent'] = coding_counts['rows'] / len(coding) * 100
display(coding_counts.round(2))

duplicate_coding = coding[coding['video_id'].duplicated(keep=False)].sort_values('video_id')
print('Manual-coding rows with a repeated video ID:', len(duplicate_coding))
display(duplicate_coding)

,rows,percent
related_clean,,
Y,314,80.10
N,78,19.90


Manual-coding rows with a repeated video ID: 8


,video_id,author_name,my_desc,related,related_clean
220,7069001683103124742,dasmandia,Ukrainian civilian talking to Russian soldiers...,Y,Y
221,7069001683103124742,dasmandia,Ukraine man talking to Russian soldiers,Y,Y
156,7075102075432176942,morriganamber,Japan taking Russian islands while invasion oc...,Y,Y
157,7075102075432176942,morriganamber,Japan taking land from Russia while Russia is ...,Y,Y
79,7075700005503536430,<NA>,Zelensky talking about the war,Y,Y
80,7075700005503536430,msnbc,Zelensky requesting aid,Y,Y
142,7077459035435437318,kira_rudik,Destruction from invasion,Y,Y
143,7077459035435437318,kira_rudik,Destruction in ukraine,Y,Y


## 6. Comments: what can and cannot be calculated

The paper reports about 4.4 million collected comments from a **subset** of videos, with collection capped at 1,000 comments per video. Those rows are not in the downloadable archive. Dividing 4.4 million by 9,503 would not be a valid mean because the numerator and denominator do not describe the same known set of videos.

Current PyTok requires a previously authenticated TikTok account. Therefore this notebook does not attempt anonymous scraping. The next cell will analyse comments automatically if you later place a hydrated `comments.csv` or `videos.csv` in this dataset directory.

In [6]:
COMMENTS_PATH = DATA_DIR / 'comments.csv'
VIDEOS_PATH = DATA_DIR / 'videos.csv'

if VIDEOS_PATH.exists():
    videos = pd.read_csv(VIDEOS_PATH, dtype={'video_id': 'string'})
    if 'comment_count' not in videos.columns:
        print('videos.csv exists, but it has no comment_count column.')
    else:
        counts = pd.to_numeric(videos['comment_count'], errors='coerce').dropna()
        display(counts.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame('platform_comment_count'))
elif COMMENTS_PATH.exists():
    comments = pd.read_csv(COMMENTS_PATH, dtype={'video_id': 'string', 'comment_id': 'string'})
    rows_per_observed_video = comments.groupby('video_id').size()
    print('Important: videos absent from comments.csv are unknown, not assumed to have zero comments.')
    display(rows_per_observed_video.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame('collected_comment_rows'))
else:
    display(pd.DataFrame([{
        'comment_analysis_status': 'Unavailable from the public download',
        'reason': 'No comments.csv or hydrated videos.csv is released; logged-in TikTok hydration is required.',
        'paper_only_aggregate': 'approximately 4.4M comments from 2.6M users and a subset of videos',
    }]))

,comment_analysis_status,reason,paper_only_aggregate
0,Unavailable from the public download,No comments.csv or hydrated videos.csv is rele...,approximately 4.4M comments from 2.6M users an...


# Findings

- The downloaded filtered release contains **9,503 unique video IDs** and **5,782 creator usernames**. It is an ID release, not the paper's full video-and-comment table.
- Every filtered ID occurs in the 18,864-ID raw collection. About half of the raw candidate IDs remain after filtering.
- A typical creator contributes one video, while a small group contributes many more. The maximum in this release is 151 videos from one username.
- The manual coding file has 392 rows but 388 unique video IDs, so four rows repeat an ID and should not automatically be treated as independent videos.
- Mean and median comments per video are **not available from the public files**. Reliable comment statistics require a completed logged-in hydration and an explicit crawl-coverage record.